# Notebook 11 — Incremental ES data, provenance & Arena-compatible session reconstruction (v0.1)

**Market Mood Forecasting · Headline Arena experiment · offline-first · default NO requests, NO purchase, NO export, NO submission**

This notebook bridges **Notebook 08** (Databento 1-minute ES research data) to **Notebook 10** (a separately frozen Logistic Regression deployment). It **never** runs or modifies the consumed Notebook 09 test, retrains the deployed model, sends an Arena prediction, or silently calls paid APIs.

### Contract and limitations

- Provider candidate: `GLBX.MDP3`, `ES.v.0` (continuous previous-session-volume selector), `ohlcv-1m`.
- Target-like session: prior calendar day **18:00 ET** to settlement date **17:00 ET** (end exclusive, last bar **16:59 ET**). Use zone-aware ET conversion.
- Historical classifier: bearish/neutral/bullish with **±0.30%** (not a universal Arena rule; every new challenge must be independently checked).
- **Only previously completed and manually reviewed eligible sessions** may become feature sources. Excluded sessions never feed Notebook 10's features.
- Continuous contracts are not guaranteed identical to the Arena front contract. Date-effective underlying-symbol resolution, contract change inspection and comparison against the actual Arena contract remain required.
- **Boundary presence ≠ full minute coverage**. Gaps/shortened sessions must be audited. Review flags are **never automatically asserted**.
- Provenance time `data_available_utc` means **the time THIS local reconstructed data was actually available to this notebook**, not a claim about historical vendor publication times. It is deliberately conservative.
- A CSV produced here does **not** authorize live submission. Notebook 10 additionally requires an independently obtained, genuine Arena challenge snapshot, settlement criteria, threshold and deadline.

### Safe staged workflow

1. **Run All unchanged**: inspect sources, quote plan, perform offline unit tests. Everything remote and all writes off.
2. Select local DBN files and, if needed, enable **free symbology metadata resolution**; inspect reconstructed candidate sessions.
3. Optionally enable **quote-only** Databento metadata. Paid access needs a **separate explicit approval** after inspecting the quote.
4. Review coverage row-by-row, supply the manual review CSV, and export **one immutable snapshot** only when all audit gates pass.
5. Hand the CSV to Notebook 10 for an **offline preview**, after checking an authoritative challenge contract.

**Never put a Databento API key in the notebook.** Keep it in the local `.env` and do not stage local licensed data or purchase logs for public Git.

## 1 · Safety configuration — read carefully

**Normal first run:** leave all switches below `False`, file lists empty, and confirmations empty. The no-network first run is intentional, not a failure.

To load data already purchased in Notebook 08, specify its **actual local DBN filename(s)** in `HISTORICAL_CONTEXT_DBN_FILES`. Use the offline catalog in Section 3 to find them. Add any new incremental DBN in `INCREMENTAL_DBN_FILES`. We will **not** assume that Notebook 08's final modeling parquet contains full OHLCV or that a DBN filename proves data coverage.

When quoting/download is requested, keep the scope small, bounded, and review the estimated cost. A quote is not purchase authorization.

In [1]:
# === USER CONFIGURATION: defaults are deliberately inert ===
DATASET = "GLBX.MDP3"
CONTINUOUS_SYMBOL = "ES.v.0"
STYPE_IN = "continuous"
BAR_SCHEMA = "ohlcv-1m"
ET_ZONE = "America/New_York"
HISTORICAL_CUTOFF = "2026-09-30"
FROZEN_PARQUET_SHA256 = "6787e64fe1c0c3508d68cd2d98cc554c6e5da19f320c7080ad7e34831b9b5db5"

# Include a 2+ session overlap with the frozen data to prove continuity.
REQUEST_START_UTC = "2026-10-22T21:00:00Z"
# Calculated from last completed normal 17:00 ET boundary; bounded to <= now.
REQUEST_END_UTC = "2026-10-23T21:00:00Z"
MAX_REQUEST_DAYS = 45

# Optional PAID/NETWORK stages. ALL remain False initially.
ENABLE_METADATA_QUOTE = False            # free metadata inquiry; requires .env key
ALLOW_PAID_DOWNLOAD = False             # potential charge; gated on preserved quote
PAID_DOWNLOAD_CONFIRMATION = ("")                   # must be exactly "I_AUTHORIZE_THIS_DATABENTO_PURCHASE_V0_1"
APPROVED_QUOTE_SHA256 = ("")              # SHA-256 printed by successful cost-quote cell
MAX_AUTHORIZED_COST_USD = 0.10            # independent hard cost ceiling

# Local-only processing. Fill paths after inspecting the offline file catalog.
HISTORICAL_CONTEXT_DBN_FILES = [
    "data/raw/databento/GLBX-20261006-RVP5LPX9NY/glbx-mdp3-20260101-20260930.ohlcv-1m.dbn.zst"
]        # e.g. ["data/raw/databento/<jobid>/<file>.dbn.zst"]
INCREMENTAL_DBN_FILES = [
    "data/raw/databento/incremental_v0_1/es_incremental_43267cc70d1ee33d.dbn.zst",
    "data/raw/databento/incremental_v0_1/es_incremental_60cf5933eecc56c0.dbn.zst",
    "data/raw/databento/incremental_v0_1/es_incremental_32292e621dbe3c76.dbn.zst",
    "data/raw/databento/incremental_v0_1/es_incremental_bf495c8a9f69de19.dbn.zst",
    "data/raw/databento/incremental_v0_1/es_incremental_6372a3ba0adcd1b3.dbn.zst",
]               # e.g. ["data/raw/databento/incremental_v0_1/<file>.dbn.zst"]
DATE_EFFECTIVE_MAPPING_CSV = ""          # optional verified columns: instrument_id,d0,d1,raw_symbol
ENABLE_SYMBOLOGY_RESOLVE = True         # free Databento metadata ONLY, explicit opt-in

# Human coverage review: never auto-set "coverage_reviewed" to True.
WRITE_UNREVIEWED_AUDIT_TEMPLATE =  False  # save immutable template AFTER DBN reconstruction
REVIEWED_AUDIT_CSV = ()               # separate, manually completed review file
EXPORT_REVIEWED_SESSION_SNAPSHOT = False
EXPORT_CONFIRMATION = ()               # "I_REVIEWED_THE_ES_SESSIONS_V0_1"

MIN_RECONCILIATION_OVERLAP = 2
MIN_ELIGIBLE_CONTEXT_SESSIONS = 20
# A diagnostic requiring investigation, NOT a replacement for Notebook 08 policy.
COVERAGE_REVIEW_THRESHOLD_PCT = 99.5

print("SAFE DEFAULTS: quote",ENABLE_METADATA_QUOTE,"purchase",ALLOW_PAID_DOWNLOAD,
      "symbology network",ENABLE_SYMBOLOGY_RESOLVE,"export",EXPORT_REVIEWED_SESSION_SNAPSHOT)
print("Notebook 08 and Notebook 09 frozen inputs will not be overwritten.")

SAFE DEFAULTS: quote False purchase False symbology network True export True
Notebook 08 and Notebook 09 frozen inputs will not be overwritten.


## 2 · Environment and frozen-upstream identity

This section uses only local files. No Databento client is constructed until you explicitly enable a network stage. **Kernel identity** is checked, because notebooks previously had incorrect-kernel problems on Windows.

In [2]:
from pathlib import Path
from datetime import datetime, timedelta, timezone
from zoneinfo import ZoneInfo
import hashlib, json, os, re, sys, subprocess
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name.lower() == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
RAW_ROOT = PROJECT_ROOT / "data" / "raw" / "databento"
PROCESSED_ROOT = PROJECT_ROOT / "data" / "processed"
NB11_ROOT = PROCESSED_ROOT / "es_arena_nb11_v0_1"
FROZEN_PARQUET = PROCESSED_ROOT / "es_daily_modeling_v0_1.parquet"
NB10_PATH = PROJECT_ROOT / "notebooks" / "10_es_arena_prospective_inference_v0_1.ipynb"
ET = ZoneInfo(ET_ZONE)

actual_python = Path(sys.executable).resolve()
expected_win = PROJECT_ROOT / ".venv" / "Scripts" / "python.exe"
expected_unix = PROJECT_ROOT / ".venv" / "bin" / "python"
if expected_win.exists() or expected_unix.exists():
    expected = expected_win if expected_win.exists() else expected_unix
    assert os.path.normcase(str(actual_python)) == os.path.normcase(str(expected.resolve())), (
        f"Wrong Jupyter kernel: {actual_python}; choose {expected}")

def require(condition, message):
    if not bool(condition):
        raise RuntimeError(message)

def sha256_file(p):
    h = hashlib.sha256()
    with Path(p).open('rb') as f:
        for chunk in iter(lambda: f.read(1024*1024),b''):
            h.update(chunk)
    return h.hexdigest()

def canonical_bytes(obj):
    return (json.dumps(obj,sort_keys=True,indent=2,ensure_ascii=False,allow_nan=False)+'\n').encode('utf-8')

def write_once(path, payload):
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)
    try:
        with path.open('xb') as f: f.write(payload)
        return 'CREATED'
    except FileExistsError:
        require(path.read_bytes()==payload, f"Refusing to overwrite existing non-identical file: {path}")
        return 'IDENTICAL EXISTING FILE'

def assert_git_ignored(path):
    p=Path(path).resolve()
    require(p.is_relative_to(PROJECT_ROOT),f'Expected project-local path: {p}')
    probe=subprocess.run(['git','check-ignore','-q',str(p)],cwd=PROJECT_ROOT,
                         capture_output=True,text=True,encoding='utf-8')
    require(probe.returncode==0,
            f'Private market/forecast output is not covered by .gitignore: {p}')

def utc_timestamp(value,name):
    t=pd.Timestamp(value)
    require(t.tzinfo is not None,f"{name} must have an explicit UTC timezone")
    return t.tz_convert('UTC')

require(FROZEN_PARQUET.is_file(),f"Frozen Notebook 08 parquet missing: {FROZEN_PARQUET}")
require(sha256_file(FROZEN_PARQUET)==FROZEN_PARQUET_SHA256,"FROZEN PARQUET CHANGED — STOP")
require(NB10_PATH.is_file(),"Notebook 10 source missing")
require(DATASET=="GLBX.MDP3" and CONTINUOUS_SYMBOL=="ES.v.0" and BAR_SCHEMA=="ohlcv-1m",
        "Provider contract differs from Notebook 08")
print("Project root:",PROJECT_ROOT)
print("Python:",actual_python)
print("Frozen historical parquet SHA-256: PASS")
print("Notebook 10 found:",NB10_PATH.name)
print("NETWORK / PAYMENT: no calls have occurred")

Project root: C:\Users\melny\Market-Mood-Forecasting
Python: C:\Users\melny\Market-Mood-Forecasting\.venv\Scripts\python.exe
Frozen historical parquet SHA-256: PASS
Notebook 10 found: 10_es_arena_prospective_inference_v0_1.ipynb
NETWORK / PAYMENT: no calls have occurred


## 3 · Local file catalog (read-only)

The following shows **filenames and sizes only**; a filename is not proof that a file contains the required dates. Choose only the relevant **recent** Notebook 08 batch files for historical context and the incremental download when available. This step never loads the entire 2010–2026 archive.

In [3]:
if not RAW_ROOT.exists():
    print("No raw Databento directory found yet:",RAW_ROOT)
    local_dbn_catalog=pd.DataFrame(columns=['relative_path','size_mb'])
else:
    candidates=sorted([p for p in RAW_ROOT.rglob('*') if p.is_file() and
                       (p.name.lower().endswith('.dbn.zst') or p.name.lower().endswith('.dbn'))])
    local_dbn_catalog=pd.DataFrame([{'relative_path':p.relative_to(PROJECT_ROOT).as_posix(),
                                    'size_mb':round(p.stat().st_size/1e6,2)} for p in candidates])
    print('Local DBN files discovered:',len(candidates))
    display(local_dbn_catalog.tail(35))
print('Selected historical context files:',len(HISTORICAL_CONTEXT_DBN_FILES))
print('Selected incremental files:',len(INCREMENTAL_DBN_FILES))
print('Reminder: use ACTUAL paths copied from the catalog, not guessed filenames.')

Local DBN files discovered: 30


,relative_path,size_mb
0,data/raw/databento/arena_06bdca67-a2d1-4356-96...,0.02
1,data/raw/databento/arena_097db04f-3dc6-49f6-90...,0.02
2,data/raw/databento/arena_167c568c-3bbc-43e4-91...,0.02
3,data/raw/databento/arena_2f41bc55-2448-4955-9f...,0.02
4,data/raw/databento/arena_46200d1b-2d69-40a8-b2...,0.02
5,data/raw/databento/arena_82629b32-f54a-460f-ae...,0.02
6,data/raw/databento/arena_b4c1d697-57ea-44a4-b7...,0.02
7,data/raw/databento/arena_e4643e45-13a6-43d1-aa...,0.02
8,data/raw/databento/GLBX-20261006-RVP5LPX9NY/gl...,2.52
9,data/raw/databento/GLBX-20261006-RVP5LPX9NY/gl...,4.59


Selected historical context files: 1
Selected incremental files: 5
Reminder: use ACTUAL paths copied from the catalog, not guessed filenames.


## 4 · Bounded incremental request plan (no network)

The start has a small overlap with the frozen history. By default the end is the **most recently completed normal 17:00 ET boundary**, which avoids buying incomplete current sessions. **Holiday and early-close rules still require manual review**. Provider availability must also be checked if you enable quoting.

The end is exclusive, so data after the end is never requested.

In [4]:
def last_normal_completed_end(now_utc,lag_minutes=90):
    now=utc_timestamp(now_utc,'now_utc')
    gate=now-pd.Timedelta(minutes=lag_minutes)
    day=gate.tz_convert(ET).date()
    for offset in range(14):
        d=day-timedelta(days=offset)
        if d.weekday()>=5: continue
        end=pd.Timestamp(f"{d.isoformat()} 17:00:00",tz=ET).tz_convert('UTC')
        if end<=gate: return end
    raise RuntimeError('Cannot find completed normal settlement boundary')

PLANNED_AT_UTC=pd.Timestamp.now(tz='UTC')
REQUEST_START=utc_timestamp(REQUEST_START_UTC,'REQUEST_START_UTC')
REQUEST_END=(utc_timestamp(REQUEST_END_UTC,'REQUEST_END_UTC') if REQUEST_END_UTC
             else last_normal_completed_end(PLANNED_AT_UTC))
require(REQUEST_START<REQUEST_END<=PLANNED_AT_UTC,'Request times invalid or end is in the future')
require((REQUEST_END-REQUEST_START)<=pd.Timedelta(days=MAX_REQUEST_DAYS),
        'Request exceeds the frozen max-day limit; shorten it deliberately')
require(REQUEST_START>=pd.Timestamp('2026-09-21T00:00:00Z'),
        'Do not buy the full-history window again')
REQUEST_ARGS={'dataset':DATASET,'symbols':CONTINUOUS_SYMBOL,'stype_in':STYPE_IN,
              'schema':BAR_SCHEMA,'start':REQUEST_START.isoformat(),'end':REQUEST_END.isoformat()}
REQUEST_ID=hashlib.sha256(canonical_bytes(REQUEST_ARGS)).hexdigest()[:16]
print('REQUEST PLAN ONLY — NOTHING DOWNLOADED')
print('Start UTC:',REQUEST_START.isoformat())
print('End UTC (exclusive):',REQUEST_END.isoformat())
print('Span days:',round((REQUEST_END-REQUEST_START).total_seconds()/86400,2))
print('Request ID:',REQUEST_ID)
print('Includes overlap with historical cutoff:',REQUEST_START<pd.Timestamp(HISTORICAL_CUTOFF,tz='UTC')+pd.Timedelta(days=1))
print('NOTE: exchange closures, shortened days, and late provider availability still need review.')

REQUEST PLAN ONLY — NOTHING DOWNLOADED
Start UTC: 2026-10-22T21:00:00+00:00
End UTC (exclusive): 2026-10-23T21:00:00+00:00
Span days: 1.0
Request ID: 6372a3ba0adcd1b3
Includes overlap with historical cutoff: False
NOTE: exchange closures, shortened days, and late provider availability still need review.


## 5 · Optional metadata-only price quote — never a purchase

Leave `ENABLE_METADATA_QUOTE=False` for an offline Run All. Turning it on calls Databento metadata endpoints (cost, count, size, dataset range) and writes **one immutable quote file**, not market data. It requires a valid `.env` entry `DATABENTO_API_KEY`.

Provider cost quotes are estimates, not authorization. Review the scope and cost **before** touching the paid-download switch.

In [5]:
def make_databento_client():
    from dotenv import load_dotenv
    load_dotenv(PROJECT_ROOT/'.env')
    require(bool(os.getenv('DATABENTO_API_KEY')), 'DATABENTO_API_KEY missing from .env')
    import databento as db
    return db.Historical()

def quote_incremental(client, request_args):
    # Databento metadata endpoints; no market-data request.
    provider_range=client.metadata.get_dataset_range(dataset=DATASET)
    available_end=utc_timestamp(provider_range['end'],'provider available_end')
    require(REQUEST_END<=available_end,
            f'Planned end {REQUEST_END} later than provider available end {available_end}')
    cost=float(client.metadata.get_cost(**request_args))
    count=int(client.metadata.get_record_count(**request_args))
    size=int(client.metadata.get_billable_size(**request_args))
    require(np.isfinite(cost) and cost>=0 and count>0 and size>0,'Bad cost/count/size quote')
    return {'request':request_args,'request_id':REQUEST_ID,
            'quoted_at_utc':pd.Timestamp.now(tz='UTC').isoformat(),
            'provider_available_end_utc':available_end.isoformat(),
            'estimated_cost_usd':cost,'estimated_records':count,'estimated_billable_bytes':size}

cost_quote=None
if not ENABLE_METADATA_QUOTE:
    print('SKIPPED: metadata quote OFF; zero Databento network calls')
else:
    cost_quote=quote_incremental(make_databento_client(),REQUEST_ARGS)
    quote_bytes=canonical_bytes(cost_quote)
    quote_hash=hashlib.sha256(quote_bytes).hexdigest()
    quote_path=NB11_ROOT/'quotes'/f'quote_{quote_hash}.json'
    assert_git_ignored(quote_path)
    print('Quote write-once:',write_once(quote_path,quote_bytes))
    print('Estimated USD:',cost_quote['estimated_cost_usd'])
    print('Estimated records:',cost_quote['estimated_records'])
    print('Estimated MB:',round(cost_quote['estimated_billable_bytes']/1e6,2))
    print('APPROVED_QUOTE_SHA256 to use AFTER YOUR REVIEW:',quote_hash)
    print('Quote path:',quote_path)
    print('THIS DID NOT PURCHASE DATA')

SKIPPED: metadata quote OFF; zero Databento network calls


## 6 · Paid download gate — deliberately disabled

**Important:** This single call could charge money. Never enable it until you explicitly decide to purchase this exact quote. Safeguards include: same quoted request, <= 2-hour quote age, maximum USD ceiling, exact confirmation phrase, the source output not already existing, and a **write-once purchase attempt marker created before the API call**.

If an API call errors after that marker, **do not blindly rerun**; you may already have been charged. Investigate the Databento request before deciding what to do. Only request `ohlcv-1m`, not trades or full history.

In [6]:
def execute_one_time_paid_download(approved_hash):
    require(approved_hash and re.fullmatch('[0-9a-f]{64}',approved_hash),
            'Provide the exact reviewed quote SHA-256')
    quote_path=NB11_ROOT/'quotes'/f'quote_{approved_hash}.json'
    require(quote_path.is_file(),'The reviewed quote JSON is missing')
    require(sha256_file(quote_path)==approved_hash,'Quote file was modified')
    q=json.loads(quote_path.read_text(encoding='utf-8'))
    require(q['request']==REQUEST_ARGS and q['request_id']==REQUEST_ID,
            'Changed request since quote — STOP; obtain new quote')
    age=pd.Timestamp.now(tz='UTC')-utc_timestamp(q['quoted_at_utc'],'quoted_at_utc')
    require(pd.Timedelta(0)<=age<=pd.Timedelta(hours=2),'Quote expired (>2 hours); re-quote')
    require(0<=float(q['estimated_cost_usd'])<=float(MAX_AUTHORIZED_COST_USD),
            'Estimated cost exceeds your explicit maximum')
    require(PAID_DOWNLOAD_CONFIRMATION=='I_AUTHORIZE_THIS_DATABENTO_PURCHASE_V0_1',
            'Missing exact cost/purchase confirmation')
    download_dir=RAW_ROOT/'incremental_v0_1'
    assert_git_ignored(download_dir/'purchase_probe.dbn.zst')
    output=download_dir/f'es_incremental_{REQUEST_ID}.dbn.zst'
    attempt=download_dir/f'PURCHASE_ATTEMPT_{REQUEST_ID}.json'
    partial=download_dir/f'es_incremental_{REQUEST_ID}.partial.dbn.zst'
    require(not output.exists() and not attempt.exists() and not partial.exists(),
            'Downloaded, partial or purchase-attempt file exists; NEVER silently purchase twice')
    marker={'request':REQUEST_ARGS,'approved_quote_sha256':approved_hash,
            'approval_cost_ceiling_usd':MAX_AUTHORIZED_COST_USD,
            'attempt_created_at_utc':pd.Timestamp.now(tz='UTC').isoformat(),
            'warning':'If request fails after this marker, do not retry without provider reconciliation'}
    download_dir.mkdir(parents=True,exist_ok=True)
    with attempt.open('xb') as f: f.write(canonical_bytes(marker))
    print('PAID CALL STARTING: attempt marker preserved:',attempt)
    client=make_databento_client()
    # Databento historical get_range(path=...) streams DBN to disk.
    client.timeseries.get_range(**REQUEST_ARGS,path=str(partial))
    require(partial.is_file() and partial.stat().st_size>0,
            'API returned without a nonempty downloaded DBN; investigate attempt')
    # Check DBN before exposing it as a final input.
    import databento as db
    db.DBNStore.from_file(partial)
    partial.rename(output)
    print('PAID DOWNLOAD COMPLETED; do not request again:',output)
    print('DBN SHA-256:',sha256_file(output))
    return output

if not ALLOW_PAID_DOWNLOAD:
    print('SKIPPED: paid-download switch OFF. No paid call.')
else:
    purchased_path=execute_one_time_paid_download(APPROVED_QUOTE_SHA256)
    print('Add this project-relative path to INCREMENTAL_DBN_FILES on next run:')
    print(purchased_path.relative_to(PROJECT_ROOT).as_posix())

SKIPPED: paid-download switch OFF. No paid call.


## 7 · Select local files and load minute bars — no API

Include enough historical context to construct **20 eligible completed sessions**, often requiring recent data from the previously purchased Notebook 08 archive plus a new incremental DBN. Notebook 08's 16-column model parquet is **not** an OHLCV context file.

No file is loaded while both user lists remain empty. When file paths are provided, each must exist **under `data/raw/databento/`**. Duplicate overlapping bars are permitted **only when every market value and instrument ID agree**; otherwise the notebook stops.

Source-file hashes and exact load timestamp are retained. Loading a local DBN does not incur Databento market-data charges.

In [7]:
def resolve_raw_path(name):
    p=(PROJECT_ROOT/str(name)).resolve()
    require(p.is_relative_to(RAW_ROOT.resolve()), f'DBN file is outside raw data directory: {p}')
    require(p.is_file(), f'DBN file not found: {p}')
    require(p.name.lower().endswith(('.dbn','.dbn.zst')),f'Unexpected DBN suffix: {p}')
    return p

def load_local_minutes(names):
    require(bool(names),'No DBN input paths selected')
    import databento as db
    paths=[resolve_raw_path(n) for n in names]
    require(len(paths)==len(set(paths)),'Duplicate input paths')
    acquired_utc=pd.Timestamp.now(tz='UTC')
    frames=[];manifest=[]
    for p in paths:
        store=db.DBNStore.from_file(p)
        frame=store.to_df(pretty_ts=True,map_symbols=False,tz='UTC')
        require(not frame.empty,f'Empty DBN: {p}')
        frame=frame.copy()
        require(isinstance(frame.index,pd.DatetimeIndex) and frame.index.tz is not None,
                f'No timezone-aware bar index: {p}')
        frame.index=frame.index.tz_convert('UTC')
        require({'open','high','low','close','volume','instrument_id'}.issubset(frame),
                f'DBN lacks OHLCV/instrument_id: {p}')
        frame=frame[['open','high','low','close','volume','instrument_id']].copy()
        frame=frame.loc[(frame.index>=pd.Timestamp('2026-08-01T00:00:00Z')) &
                        (frame.index<REQUEST_END)]
        if frame.empty: continue
        frame['source_file']=p.relative_to(PROJECT_ROOT).as_posix()
        frames.append(frame)
        manifest.append({'path':p.relative_to(PROJECT_ROOT).as_posix(),'bytes':p.stat().st_size,
                         'sha256':sha256_file(p),'loaded_at_utc':acquired_utc.isoformat(),
                         'first_bar_utc':frame.index.min().isoformat(),
                         'last_bar_utc':frame.index.max().isoformat(),'rows_selected':len(frame)})
    require(frames,'Selected DBNs have no 2026-08 onward complete-window data')
    all_bars=pd.concat(frames).sort_index()
    for col in ['open','high','low','close','volume','instrument_id']:
        all_bars[col]=pd.to_numeric(all_bars[col],errors='raise')
    require(np.isfinite(all_bars[['open','high','low','close','volume','instrument_id']].to_numpy(dtype=float)).all(),
            'Nonfinite bar price, volume, or instrument ID')
    require(((all_bars[['open','high','low','close']]>0).all(axis=1)).all(),'Nonpositive ES bar price')
    require((all_bars.volume>=0).all(),'Negative bar volume')
    require(all_bars.index.floor('min').equals(all_bars.index),'Bar index not minute aligned')
    if all_bars.index.has_duplicates:
        dup_rows=all_bars.loc[all_bars.index.duplicated(keep=False)]
        for stamp,g in dup_rows.groupby(level=0):
            v=g[['open','high','low','close','volume','instrument_id']].to_numpy()
            require(np.all(v==v[0]),f'Conflicting overlapping source bars at {stamp}')
        all_bars=all_bars.loc[~all_bars.index.duplicated(keep='first')].copy()
    require(all_bars.index.is_unique,'Minute timestamps not unique')
    require(all_bars.index.is_monotonic_increasing,'Minute bars not sorted')
    # Preserve an immutable first-observed timestamp for reproducible reviewer CSVs.
    # A new notebook run must not silently change data_available_utc.
    source_ident={'files':sorted([{'path':x['path'],'sha256':x['sha256'],
                  'bytes':x['bytes']} for x in manifest],key=lambda x:x['path'])}
    source_hash=hashlib.sha256(canonical_bytes(source_ident)).hexdigest()[:20]
    first_seen_path=NB11_ROOT/'source_ingestion'/f'first_seen_{source_hash}.json'
    assert_git_ignored(first_seen_path)
    if first_seen_path.is_file():
        first_seen=json.loads(first_seen_path.read_text(encoding='utf-8'))
        require(first_seen.get('source_files')==source_ident,
                'Previously saved first-observed source identity changed')
        acquired_utc=utc_timestamp(first_seen['first_observed_at_utc'],
                                  'first_observed_at_utc')
    else:
        first_seen={'source_files':source_ident,'first_observed_at_utc':acquired_utc.isoformat(),
                    'semantics':'First observation by this Notebook 11; no historical vendor availability claim'}
        print('FIRST-OBSERVED SOURCE MANIFEST:',write_once(first_seen_path,canonical_bytes(first_seen)))
    for item in manifest:
        item['loaded_at_utc']=acquired_utc.isoformat()  # stable first-observed provenance
    return all_bars,pd.DataFrame(manifest),acquired_utc

selected_files=HISTORICAL_CONTEXT_DBN_FILES+INCREMENTAL_DBN_FILES
minute_bars=None;local_source_manifest=None;source_acquired_utc=None
if not selected_files:
    print('SKIPPED: no local DBN files selected. This is expected on the first run.')
else:
    minute_bars,local_source_manifest,source_acquired_utc=load_local_minutes(selected_files)
    print('LOCAL MINUTE HISTORY:',len(minute_bars),'bars; first:',minute_bars.index.min(),
          'last:',minute_bars.index.max())
    display(local_source_manifest)

LOCAL MINUTE HISTORY: 82548 bars; first: 2026-08-02 22:00:00+00:00 last: 2026-10-23 20:59:00+00:00


,path,bytes,sha256,loaded_at_utc,first_bar_utc,last_bar_utc,rows_selected
0,data/raw/databento/GLBX-20261006-RVP5LPX9NY/gl...,4181250,94d7b731ac96fe818862ad673f0230d231697072ddd79a...,2026-10-26T09:51:11.499598+00:00,2026-08-02T22:00:00+00:00,2026-09-30T20:59:00+00:00,59088
1,data/raw/databento/incremental_v0_1/es_increme...,284398,eea7efb5381860bdc7bd1a07c996785162ef9914556b9d...,2026-10-26T09:51:11.499598+00:00,2026-09-30T22:00:00+00:00,2026-10-19T20:59:00+00:00,17940
2,data/raw/databento/incremental_v0_1/es_increme...,23531,20c2a4b85e596f62f119c7a70ef3c4f42f0b2846a7353b...,2026-10-26T09:51:11.499598+00:00,2026-10-19T22:00:00+00:00,2026-10-20T20:59:00+00:00,1380
3,data/raw/databento/incremental_v0_1/es_increme...,22232,568789981d5e0c2dc7715047c120fc9d1421f52a0a92a7...,2026-10-26T09:51:11.499598+00:00,2026-10-20T22:00:00+00:00,2026-10-21T20:59:00+00:00,1380
4,data/raw/databento/incremental_v0_1/es_increme...,23665,1523bb695547409a2ce0e72ac023d5d1d9a704112da71a...,2026-10-26T09:51:11.499598+00:00,2026-10-21T22:00:00+00:00,2026-10-22T20:59:00+00:00,1380
5,data/raw/databento/incremental_v0_1/es_increme...,23136,c7f8367b60da2fe82894c7b8da105e2f16b5d09322673c...,2026-10-26T09:51:11.499598+00:00,2026-10-22T22:00:00+00:00,2026-10-23T20:59:00+00:00,1380


## 8 · Date-effective ES futures contract mapping

Databento reuses numeric `instrument_id` across time. **Never assign one symbol to an ID globally.** We require a date-effective mapping with half-open `[d0, d1)` validity intervals. The local mapping CSV must have `instrument_id,d0,d1,raw_symbol`. Alternatively, turn on `ENABLE_SYMBOLOGY_RESOLVE` to request free provider metadata; no market data is purchased. Date-effective mapping validity is checked for **each individual bar**.

**Do not guess a front-contract symbol.** We later verify the most recent raw contract against Kopei's actual reviewed Arena challenge.

In [8]:
CONTRACT_RE=re.compile(r'^ES[HMUZ][0-9]{1,2}$')

def provider_mapping_for_bars(client,bars):
    ids=sorted(set(bars.instrument_id.astype('int64')))
    start=bars.index.min().date().isoformat()
    end=(bars.index.max().normalize()+pd.Timedelta(days=1)).date().isoformat()

    print("SYMBOL MAPPING START:", start)
    print("SYMBOL MAPPING END (exclusive):", end)
    
    result=[]
    for j in range(0,len(ids),200):
        sub=ids[j:j+200]
        payload=client.symbology.resolve(dataset=DATASET,symbols=[int(x) for x in sub],
                       stype_in='instrument_id',stype_out='raw_symbol',start_date=start,end_date=end)
        require(int(payload.get('status',2)) in (0,1),'Symbology resolution status is failure')
        require(not payload.get('not_found',[]),'Databento could not resolve instrument IDs')
        for id_ in sub:
            for entry in payload.get('result',{}).get(str(id_),[]):
                result.append({'instrument_id':id_,'d0':entry['d0'],'d1':entry['d1'],'raw_symbol':entry['s']})
    require(bool(result),'Provider symbology returned no intervals')
    return pd.DataFrame(result)

def attach_date_effective_symbols(bars,mapping):
    needed={'instrument_id','d0','d1','raw_symbol'}
    require(needed.issubset(mapping),f'Symbology missing columns: {needed-set(mapping)}')
    m=mapping[list(needed)].copy()
    m['instrument_id']=pd.to_numeric(m.instrument_id,errors='raise').astype('int64')
    m['d0']=pd.to_datetime(m.d0,utc=True,errors='raise').dt.normalize()
    m['d1']=pd.to_datetime(m.d1,utc=True,errors='raise').dt.normalize()
    m['raw_symbol']=m.raw_symbol.astype(str)
    m=m.loc[m.raw_symbol.str.fullmatch(CONTRACT_RE)].drop_duplicates()
    require(not m.empty,'No eligible quarterly ES symbol mappings')
    require((m.d0<m.d1).all(),'Invalid symbology date intervals')
    bars=bars.copy()
    require(bars.index.tz is not None,'Bar timestamps must be timezone aware')
    # Vectorized as-of mapping per instrument using UTC mapping validity days.
    resolved=[]
    for instrument_id,g in bars.groupby('instrument_id'):
        intervals=m.loc[m.instrument_id==int(instrument_id)].sort_values('d0').copy()
        require(not intervals.empty,f'Missing effective mapping for instrument {instrument_id}')
        require((intervals.d0.iloc[1:].to_numpy() >= intervals.d1.iloc[:-1].to_numpy()).all(),
                f'Overlapping map intervals for instrument {instrument_id}')
        gg=g.copy()
        ts=pd.DatetimeIndex(gg.index).tz_convert('UTC').normalize()
        intervals_start=pd.DatetimeIndex(intervals.d0)
        idx=intervals_start.searchsorted(ts,side='right')-1
        require((idx>=0).all(),f'Unresolved early bar dates for {instrument_id}')
        selected=intervals.iloc[idx]
        require((ts.to_numpy() < pd.DatetimeIndex(selected.d1).to_numpy()).all(),
                f'Bar outside symbology validity interval for {instrument_id}')
        gg['raw_symbol']=selected.raw_symbol.to_numpy()
        resolved.append(gg)
    out=pd.concat(resolved).sort_index()
    require(len(out)==len(bars) and out.index.equals(bars.index),'Symbology changed bar cardinality')
    require(out.raw_symbol.map(lambda s: bool(CONTRACT_RE.fullmatch(str(s)))).all(),'Non-outright symbol')
    return out

resolved_bars=None; mapping_table=None
if minute_bars is None:
    print('SKIPPED: symbology requires locally loaded DBN minute bars.')
elif DATE_EFFECTIVE_MAPPING_CSV:
    mapping_path=(PROJECT_ROOT/DATE_EFFECTIVE_MAPPING_CSV).resolve()
    require(mapping_path.is_file() and mapping_path.is_relative_to(PROJECT_ROOT),
            'Manual date-effective mapping CSV is missing/outside project')
    mapping_table=pd.read_csv(mapping_path)
    resolved_bars=attach_date_effective_symbols(minute_bars,mapping_table)
    print('Applied manual source mapping:',mapping_path,'SHA-256',sha256_file(mapping_path))
elif ENABLE_SYMBOLOGY_RESOLVE:
    mapping_table=provider_mapping_for_bars(make_databento_client(),minute_bars)
    resolved_bars=attach_date_effective_symbols(minute_bars,mapping_table)
    print('Free symbology metadata used; symbols resolved from provider validity windows.')
else:
    print('BLOCKED (safe): specify DATE_EFFECTIVE_MAPPING_CSV or enable free symbology lookup.')
if resolved_bars is not None:
    print('DATE-EFFECTIVE RAW CONTRACTS VERIFIED:',resolved_bars.raw_symbol.value_counts().to_dict())

SYMBOL MAPPING START: 2026-08-02
SYMBOL MAPPING END (exclusive): 2026-10-24
Free symbology metadata used; symbols resolved from provider validity windows.
DATE-EFFECTIVE RAW CONTRACTS VERIFIED: {'ESU6': 44028, 'ESZ6': 38520}


## 9 · Reconstruct completed 18:00–17:00 ET sessions

For modern ES sessions, expected bars are **1,380** per normal full session. The exact first **18:00** and last **16:59** must exist. The model's historical `target_eligible` structural rule is preserved, while diagnostics show missing minutes and low coverage.

**No `coverage_reviewed=True` is assigned automatically**. Reconstructed rows initially have this field `False`, requiring explicit human sign-off. Sessions incomplete at the run time are not accepted as finished.

In [9]:
SESSION_EXPORT_COLUMNS=[
 'session_date','session_start_utc','session_end_utc','data_available_utc',
 'open','high','low','close','volume','session_return','raw_symbol',
 'target_eligible','first_bar_on_expected_boundary','last_bar_on_expected_boundary',
 'multiple_underlying_symbols','coverage_reviewed','source_dataset','source_schema']

def create_session_rows(bars,observed_at_utc):
    require(isinstance(bars.index,pd.DatetimeIndex) and bars.index.tz is not None,
            'Need timezone-aware minute index')
    require(bars.index.is_unique and bars.index.is_monotonic_increasing,'Expected unique sorted minute timestamps')
    require({'open','high','low','close','volume','raw_symbol'}.issubset(bars),
            'Bars missing fields needed for ES session reconstruction')
    observed=utc_timestamp(observed_at_utc,'observed_at_utc')
    ny=bars.tz_convert(ET).copy()
    # Notebook 08 groups by local ET index shifted +6 hours; matches frozen construction.
    ny['session_date']=(ny.index+pd.Timedelta(hours=6)).date
    rows=[]
    for day,all_day in ny.groupby('session_date',sort=True):
        day=pd.Timestamp(day).date()
        # Make exact ET boundaries, with timezone-aware DST handling.
        prior=day-timedelta(days=1)
        start_et=pd.Timestamp(f'{prior.isoformat()} 18:00:00',tz=ET)
        end_et=pd.Timestamp(f'{day.isoformat()} 17:00:00',tz=ET)
        last_et=pd.Timestamp(f'{day.isoformat()} 16:59:00',tz=ET)
        g=all_day.loc[(all_day.index>=start_et)&(all_day.index<end_et)].sort_index()
        if g.empty: continue
        if end_et.tz_convert('UTC')>observed: continue  # never label an unfinished session
        first_ok=bool(g.index[0]==start_et)
        last_ok=bool(g.index[-1]==last_et)
        syms=sorted(set(g.raw_symbol.astype(str)))
        many=len(syms)!=1
        op=float(g.open.iloc[0]); hi=float(g.high.max()); lo=float(g.low.min())
        cl=float(g.close.iloc[-1]);vol=float(g.volume.sum())
        valid=bool(np.isfinite([op,hi,lo,cl,vol]).all() and op>0 and cl>0 and
                   hi>=max(op,cl,lo) and lo<=min(op,cl,hi) and vol>=0)
        ret=float(cl/op-1) if valid else float('nan')
        structurally_eligible=bool(first_ok and last_ok and not many and valid and np.isfinite(ret))
        exp_bars=1380
        diffs=pd.Series(g.index).diff().dropna().dt.total_seconds()/60
        observed_bars=len(g)
        rows.append({'session_date':day.isoformat(),
           'session_start_utc':start_et.tz_convert('UTC').isoformat(),
           'session_end_utc':end_et.tz_convert('UTC').isoformat(),
           # Proof of acquisition rather than inferred historical data availability:
           'data_available_utc':observed.isoformat(),
           'open':op,'high':hi,'low':lo,'close':cl,'volume':vol,
           'session_return':ret,'raw_symbol':syms[0] if not many else '',
           'target_eligible':structurally_eligible,
           'first_bar_on_expected_boundary':first_ok,'last_bar_on_expected_boundary':last_ok,
           'multiple_underlying_symbols':many,'coverage_reviewed':False,
           'source_dataset':DATASET,'source_schema':BAR_SCHEMA,
           'bar_count':observed_bars,'expected_bar_count':exp_bars,
           'bar_coverage_pct':100*observed_bars/exp_bars,
           'bar_count_warning':observed_bars!=exp_bars,
           'max_gap_minutes':float(diffs.max()) if len(diffs) else 0.0,
           'review_flags': '|'.join(x for x,on in [
              ('missing_18_open',not first_ok),('missing_1659_close',not last_ok),
              ('multiple_contracts',many),('invalid_ohlcv',not valid),
              ('low_minute_coverage',100*observed_bars/exp_bars<COVERAGE_REVIEW_THRESHOLD_PCT),
              ('large_internal_gap',bool(len(diffs) and diffs.max()>1))] if on)})
    require(bool(rows),'No completed ES sessions reconstructed')
    s=pd.DataFrame(rows).sort_values('session_date').reset_index(drop=True)
    require(s.session_date.is_unique,'Duplicate settlement-session dates')
    return s

session_candidates=None
if resolved_bars is None:
    print('SKIPPED: session reconstruction waiting for date-resolved local DBN data.')
else:
    session_candidates=create_session_rows(resolved_bars,source_acquired_utc)
    print('Completed reconstructed sessions:',len(session_candidates))
    print('Structurally eligible:',int(session_candidates.target_eligible.sum()))
    print('Excluded:',int((~session_candidates.target_eligible).sum()))
    print('Dates:',session_candidates.session_date.iloc[0],'->',session_candidates.session_date.iloc[-1])
    display(session_candidates.tail(30))

Completed reconstructed sessions: 60
Structurally eligible: 58
Excluded: 2
Dates: 2026-08-03 -> 2026-10-23


,session_date,session_start_utc,session_end_utc,data_available_utc,open,high,low,close,volume,session_return,...,multiple_underlying_symbols,coverage_reviewed,source_dataset,source_schema,bar_count,expected_bar_count,bar_coverage_pct,bar_count_warning,max_gap_minutes,review_flags
30,2026-09-14,2026-09-13T22:00:00+00:00,2026-09-14T21:00:00+00:00,2026-10-26T09:51:11.499598+00:00,7609.25,7652.00,7593.75,7632.25,737362.0,0.003023,...,False,False,GLBX.MDP3,ohlcv-1m,1380,1380,100.000000,False,1.0,
31,2026-09-15,2026-09-14T22:00:00+00:00,2026-09-15T21:00:00+00:00,2026-10-26T09:51:11.499598+00:00,7631.00,7633.25,7576.75,7596.75,258019.0,-0.004488,...,False,False,GLBX.MDP3,ohlcv-1m,1379,1380,99.927536,True,2.0,large_internal_gap
32,2026-09-16,2026-09-15T22:00:00+00:00,2026-09-16T21:00:00+00:00,2026-10-26T09:51:11.499598+00:00,7596.50,7699.00,7575.00,7622.00,1418765.0,0.003357,...,True,False,GLBX.MDP3,ohlcv-1m,1375,1380,99.637681,True,2.0,multiple_contracts|large_internal_gap
33,2026-09-17,2026-09-16T22:00:00+00:00,2026-09-17T21:00:00+00:00,2026-10-26T09:51:11.499598+00:00,7624.00,7722.25,7617.50,7702.25,1239922.0,0.010264,...,False,False,GLBX.MDP3,ohlcv-1m,1380,1380,100.000000,False,1.0,
34,2026-09-18,2026-09-17T22:00:00+00:00,2026-09-18T21:00:00+00:00,2026-10-26T09:51:11.499598+00:00,7703.50,7739.25,7675.00,7725.00,1275838.0,0.002791,...,False,False,GLBX.MDP3,ohlcv-1m,1380,1380,100.000000,False,1.0,
35,2026-09-21,2026-09-20T22:00:00+00:00,2026-09-21T21:00:00+00:00,2026-10-26T09:51:11.499598+00:00,7722.00,7847.25,7713.75,7829.25,1289899.0,0.013889,...,False,False,GLBX.MDP3,ohlcv-1m,1380,1380,100.000000,False,1.0,
36,2026-09-22,2026-09-21T22:00:00+00:00,2026-09-22T21:00:00+00:00,2026-10-26T09:51:11.499598+00:00,7830.00,7848.50,7810.50,7827.75,1259586.0,-0.000287,...,False,False,GLBX.MDP3,ohlcv-1m,1380,1380,100.000000,False,1.0,
37,2026-09-23,2026-09-22T22:00:00+00:00,2026-09-23T21:00:00+00:00,2026-10-26T09:51:11.499598+00:00,7830.00,7843.25,7758.75,7772.75,1487166.0,-0.007312,...,False,False,GLBX.MDP3,ohlcv-1m,1380,1380,100.000000,False,1.0,
38,2026-09-24,2026-09-23T22:00:00+00:00,2026-09-24T21:00:00+00:00,2026-10-26T09:51:11.499598+00:00,7774.75,7783.50,7707.25,7759.50,1686287.0,-0.001961,...,False,False,GLBX.MDP3,ohlcv-1m,1380,1380,100.000000,False,1.0,
39,2026-09-25,2026-09-24T22:00:00+00:00,2026-09-25T21:00:00+00:00,2026-10-26T09:51:11.499598+00:00,7759.00,7814.75,7748.50,7805.75,1566409.0,0.006025,...,False,False,GLBX.MDP3,ohlcv-1m,1380,1380,100.000000,False,1.0,


## 10 · Verify continuity against frozen Notebook 08

Overlap the new reconstructed sessions with the **actual frozen parquet** before exporting. We require at least two matching eligible dates and exact-enough reconstructed returns. This catches silent changes in time boundaries, continuous-contract mapping or price aggregation.

If the latest historical source does not include two matching overlapping sessions, **add the correct Notebook 08 context DBN**; never reduce the minimum or invent overlapping rows.

In [10]:
def reconcile_to_frozen(candidates, frozen_parquet=FROZEN_PARQUET):
    frozen=pd.read_parquet(frozen_parquet)
    require({'session_date','target_return','target'}.issubset(frozen),
            'Frozen Notebook 08 parquet has unexpected schema')
    left=candidates.loc[candidates.target_eligible,['session_date','session_return']].copy()
    left.session_date=pd.to_datetime(left.session_date).dt.strftime('%Y-%m-%d')
    right=frozen[['session_date','target_return']].copy()
    right.session_date=pd.to_datetime(right.session_date).dt.strftime('%Y-%m-%d')
    right=right.loc[(right.session_date>='2026-09-21') & (right.session_date<=HISTORICAL_CUTOFF)]
    merged=left.merge(right,on='session_date',how='inner',validate='one_to_one')
    if len(merged):
        merged['abs_return_diff']=(merged.session_return-merged.target_return).abs()
    else:
        merged['abs_return_diff']=pd.Series(dtype=float)
    passed=bool(len(merged)>=MIN_RECONCILIATION_OVERLAP and
                (merged.abs_return_diff<=1e-10).all())
    return merged,passed

overlap_report=None; overlap_pass=False
if session_candidates is None:
    print('SKIPPED: frozen-2026 overlap audit awaits reconstructed sessions.')
else:
    overlap_report,overlap_pass=reconcile_to_frozen(session_candidates)
    print('Overlap source dates:',len(overlap_report))
    print('Agreement threshold:',1e-10)
    print('Minimum overlapping sessions:',MIN_RECONCILIATION_OVERLAP)
    display(overlap_report)
    print('FROZEN NB08 HISTORICAL OVERLAP:', 'PASS' if overlap_pass else 'FAIL — DO NOT EXPORT')

Overlap source dates: 8
Agreement threshold: 1e-10
Minimum overlapping sessions: 2


,session_date,session_return,target_return,abs_return_diff
0,2026-09-21,0.013889,0.013889,0.0
1,2026-09-22,-0.000287,-0.000287,0.0
2,2026-09-23,-0.007312,-0.007312,0.0
3,2026-09-24,-0.001961,-0.001961,0.0
4,2026-09-25,0.006025,0.006025,0.0
5,2026-09-28,-0.006381,-0.006381,0.0
6,2026-09-29,-0.001065,-0.001065,0.0
7,2026-09-30,-0.002261,-0.002261,0.0


FROZEN NB08 HISTORICAL OVERLAP: PASS


## 11 · Independent synthetic boundary, contract, coverage and overlap tests

No vendor requests, purchases or official challenges are involved. These tests must pass even when no local market files are configured.

In [11]:
def test_session_builder():
    # 1380 minute timestamps from Sunday 18:00 to Monday 16:59 ET (Oct 19 2026)
    start=pd.Timestamp('2026-10-18 18:00',tz=ET).tz_convert('UTC')
    ticks=pd.date_range(start,periods=1380,freq='min')
    x=pd.DataFrame({'open':5000.0,'high':5001.0,'low':4999.0,'close':5000.0,
                    'volume':10,'raw_symbol':'ESZ6'},index=ticks)
    a=create_session_rows(x,ticks[-1]+pd.Timedelta(hours=3))
    require(len(a)==1 and a.target_eligible.iloc[0] and a.bar_count.iloc[0]==1380,
            'Normal complete synthetic ES session did not qualify')
    require(a.coverage_reviewed.iloc[0]==False,'Synthetic session incorrectly auto-reviewed')
    require(a.session_date.iloc[0]=='2026-10-19','Synthetic settlement date incorrect')
    require(a.session_start_utc.iloc[0]==start.isoformat(),'Session start miscomputed')
    require(a.session_end_utc.iloc[0]==(ticks[-1]+pd.Timedelta(minutes=1)).isoformat(),
            'Session end miscomputed')
    y=create_session_rows(x.iloc[1:],ticks[-1]+pd.Timedelta(hours=3))
    require(not y.target_eligible.iloc[0],'Missing opening bar falsely accepted')
    z=create_session_rows(x.iloc[:-1],ticks[-1]+pd.Timedelta(hours=3))
    require(not z.target_eligible.iloc[0],'Missing closing bar falsely accepted')
    p=x.copy();p.loc[p.index[30:50],'raw_symbol']='ESU6'
    q=create_session_rows(p,ticks[-1]+pd.Timedelta(hours=3))
    require(not q.target_eligible.iloc[0] and q.multiple_underlying_symbols.iloc[0],
            'Mixed contract session falsely accepted')
    # Small hidden interior gaps must be flagged for human review.
    g=create_session_rows(x.drop(x.index[100]),ticks[-1]+pd.Timedelta(hours=3))
    require(g.target_eligible.iloc[0] and 'large_internal_gap' in g.review_flags.iloc[0],
            'Coverage diagnostic failed to flag missing internal minute')
    # No session may be considered completed before its 17:00 ET boundary.
    try: create_session_rows(x,ticks[-1]-pd.Timedelta(minutes=3))
    except RuntimeError: pass
    else: raise AssertionError('An unfinished synthetic session was included')
    return True

def test_mapping_intervals():
    ts=pd.to_datetime(['2026-09-15T00:00:00Z','2026-09-16T00:00:00Z'])
    bars=pd.DataFrame({'instrument_id':[55,55],'open':[10,10],'high':[11,11],
                       'low':[9,9],'close':[10,10],'volume':[5,5]},index=ts)
    mapping=pd.DataFrame({'instrument_id':[55,55],'d0':['2026-09-14','2026-09-16'],
                          'd1':['2026-09-16','2026-09-18'],'raw_symbol':['ESU6','ESZ6']})
    got=attach_date_effective_symbols(bars,mapping)
    require(got.raw_symbol.tolist()==['ESU6','ESZ6'],'Date-effective mapping failed contract roll')
    return True

print('SYNTHETIC SESSION/ROLLOVER/BAR-COVERAGE TESTS:', 'PASS' if test_session_builder() and test_mapping_intervals() else 'FAIL')

SYNTHETIC SESSION/ROLLOVER/BAR-COVERAGE TESTS: PASS


## 12 · Human coverage review and write-once exports

The **candidate audit** contains extra columns (bar count, coverage percentage, max gap, review flags). It is **not** the Notebook 10 CSV. 

To proceed:

1. Review every source session, including excluded/incomplete sessions. Match suspicious days to CME trading calendar, provider conditions and actual minute-bar coverage. Do not treat the 99.5% indicator as a conclusive coverage audit.
2. Enable `WRITE_UNREVIEWED_AUDIT_TEMPLATE=True` to save a candidate CSV with all `coverage_reviewed=False`. It is immutable and identified by the source hashes.
3. Make a **separate copy** of that CSV and manually set `coverage_reviewed=true` **only for sessions actually reviewed**, with nonblank `reviewer` and `review_notes`. Set review notes for incomplete/shortened or roll cases. Do not edit the immutable candidate file.
4. Provide that file in `REVIEWED_AUDIT_CSV`, enable export and supply the exact confirmation phrase. The export validates it against reconstructed values, including checking **every field other than the human-review fields**.
5. A minimum of **20 eligible source sessions**, 2 matching historical overlaps and at least one new eligible session after Sep 30, 2026 are mandatory. 

The final output is **an immutable, uniquely named snapshot** with the exact 18 columns Notebook 10 requires. If a canonical `es_arena_completed_sessions_v0_1.csv` is absent, the first successfully reviewed snapshot is also copied there *once*. Later snapshots are **never** allowed to overwrite it; use the dated snapshot path explicitly for future predictions. 

**Do not create a fake Arena review JSON or forecast here.**

In [12]:
def candidate_identity(candidates,files_manifest):
    # Stable identity independent of local observed_at timestamp in candidate rows.
    source=[{'path':r['path'],'sha256':r['sha256'],'bytes':int(r['bytes'])}
            for r in files_manifest.to_dict('records')]
    return hashlib.sha256(canonical_bytes({'raw_sources':sorted(source,key=lambda x:x['path']),
                                            'session_first':candidates.session_date.iloc[0],
                                            'session_last':candidates.session_date.iloc[-1],
                                            'version':'nb11-v0.1'})).hexdigest()[:16]

def csv_bytes(df):
    return df.to_csv(index=False,float_format='%.12g',lineterminator='\n').encode('utf-8')

candidate_id=None; candidate_audit_path=None
if session_candidates is not None:
    candidate_id=candidate_identity(session_candidates,local_source_manifest)
    candidate_audit_path=NB11_ROOT/'audits'/f'unreviewed_{candidate_id}.csv'
    print('Candidate audit path:',candidate_audit_path)
    if WRITE_UNREVIEWED_AUDIT_TEMPLATE:
        template=session_candidates.copy()
        template['reviewer']='';template['review_notes']='';template['coverage_reviewed']=False
        assert_git_ignored(candidate_audit_path)
        result=write_once(candidate_audit_path,csv_bytes(template))
        print('UNREVIEWED AUDIT TEMPLATE:',result)
else:
    print('SKIPPED: audit-template needs reconstructed actual DBN sessions.')

Candidate audit path: C:\Users\melny\Market-Mood-Forecasting\data\processed\es_arena_nb11_v0_1\audits\unreviewed_ab9c0a6ade744429.csv


In [13]:
def verify_reviewed_rows(candidates,reviewed):
    require(not candidates.empty,'No candidate sessions to review')
    require('session_date' in reviewed and reviewed.session_date.is_unique,
            'Review dates missing/duplicated')
    require(len(reviewed)==len(candidates) and set(reviewed.session_date)==set(candidates.session_date),
            'Review must contain precisely the reconstructed candidate session dates')
    require({'reviewer','review_notes','coverage_reviewed'}.issubset(reviewed),
            'Missing manual-review attestation fields')
    vals=reviewed.coverage_reviewed.astype(str).str.strip().str.lower()
    require(vals.isin(['true','false']).all() and vals.eq('true').all(),
            'Every source session must be individually marked reviewed=true')
    require(reviewed.reviewer.astype(str).str.strip().ne('').all() and
            reviewed.review_notes.astype(str).str.strip().ne('').all(),
            'Each reviewed session needs named reviewer and nonblank review notes')
    # Compare CSV roundtrip exactly, excluding only human-written fields.
    allowed={'reviewer','review_notes','coverage_reviewed'}
    compare_cols=[col for col in candidates.columns if col not in allowed]
    a=candidates.sort_values('session_date').reset_index(drop=True)
    b=reviewed.sort_values('session_date').reset_index(drop=True)
    require(set(compare_cols).issubset(b),'Review file deleted reconstructed data fields')
    for col in compare_cols:
        if pd.api.types.is_numeric_dtype(a[col]) and not pd.api.types.is_bool_dtype(a[col]):
            lhs=pd.to_numeric(a[col],errors='coerce').to_numpy(dtype=float)
            rhs=pd.to_numeric(b[col],errors='coerce').to_numpy(dtype=float)
            require(np.allclose(lhs,rhs,rtol=0,atol=1e-8,equal_nan=True),
                    f'Review modified original numeric data: {col}')
        else:
            require(a[col].astype(str).fillna('').tolist()==b[col].astype(str).fillna('').tolist(),
                    f'Review modified frozen candidate values: {col}')
    return a.assign(coverage_reviewed=True),b

def verify_for_nb10(candidate,now_utc=None):
    today=utc_timestamp(now_utc or pd.Timestamp.now(tz='UTC'),'now_utc')
    require(overlap_pass,'Frozen Notebook 08 overlap has not passed')
    require(candidate.session_date.is_unique and candidate.session_date.is_monotonic_increasing,
            'Session dates are duplicated or unsorted')
    eligible=candidate.loc[candidate.target_eligible]
    require(len(eligible)>=MIN_ELIGIBLE_CONTEXT_SESSIONS,
            'Insufficient 20 eligible sessions; include more purchased Notebook 08 context DBNs')
    require(eligible.session_date.max()>HISTORICAL_CUTOFF,
            'No NEW source session after 2026-09-30; incremental data is missing')
    latest_session=pd.Timestamp(eligible.session_date.max()).date()
    age_days=(today.tz_convert(ET).date()-latest_session).days
    require(0<=age_days<=7,
            f'Newest reviewed eligible session is stale ({age_days} calendar days old)')
    require(candidate.coverage_reviewed.all(),'Coverage not all reviewed')
    for row in eligible.itertuples():
        require(CONTRACT_RE.fullmatch(str(row.raw_symbol)) is not None,
                f'Invalid raw contract {row.raw_symbol}')
        require(utc_timestamp(row.session_end_utc,'session_end_utc')<=
                utc_timestamp(row.data_available_utc,'data_available_utc')<=today,
                f'No evidence session completed and was available by now: {row.session_date}')
    # For a later Arena preview, NB10 will recheck staleness and its own cutoff.
    return True

reviewed_sessions=None; exported_snapshot=None
if not EXPORT_REVIEWED_SESSION_SNAPSHOT:
    print('SKIPPED: reviewed CSV export OFF; no Notebook 10 input written')
else:
    require(session_candidates is not None and local_source_manifest is not None,
            'Need actual reconstructed local market data')
    require(EXPORT_CONFIRMATION=='I_REVIEWED_THE_ES_SESSIONS_V0_1',
            'Human-review export confirmation phrase missing')
    require(REVIEWED_AUDIT_CSV,'Specify a manually reviewed audit CSV')
    rpath=(PROJECT_ROOT/REVIEWED_AUDIT_CSV).resolve()
    require(rpath.is_file() and rpath.is_relative_to(PROJECT_ROOT),
            'Reviewed audit CSV is missing or outside project')
    reviewed_df=pd.read_csv(rpath,keep_default_na=False)
    reviewed_sessions,review_evidence=verify_reviewed_rows(session_candidates,reviewed_df)
    require(verify_for_nb10(reviewed_sessions),'Notebook 10 session evidence gate failed')
    output=reviewed_sessions.loc[:,SESSION_EXPORT_COLUMNS].copy()
    assert_git_ignored(NB11_ROOT/'snapshots'/'privacy_probe.csv')
    assert_git_ignored(PROCESSED_ROOT/'es_arena_completed_sessions_v0_1.csv')
    payload=csv_bytes(output)
    digest=hashlib.sha256(payload).hexdigest()
    last_date=output.session_date.iloc[-1].replace('-','')
    exported_snapshot=NB11_ROOT/'snapshots'/f'es_sessions_{last_date}_{digest[:16]}.csv'
    require(mapping_table is not None and not mapping_table.empty,
            'Date-effective mapping metadata not available for export')
    mapping_csv=csv_bytes(mapping_table.sort_values(['instrument_id','d0','raw_symbol']))
    mapping_sha=hashlib.sha256(mapping_csv).hexdigest()
    mapping_copy=NB11_ROOT/'symbology'/f'date_effective_{mapping_sha[:16]}.csv'
    assert_git_ignored(mapping_copy)
    print('DATED SYMBOLOGY PROVENANCE:',write_once(mapping_copy,mapping_csv))
    print('IMMUTABLE SESSION SNAPSHOT:',write_once(exported_snapshot,payload))
    export_manifest={'version':'v0.1',
        'source_first_observed_at_utc':source_acquired_utc.isoformat(),
        'target_source':'Notebook 10 validated completed ES source sessions',
        'snapshot_path':exported_snapshot.relative_to(PROJECT_ROOT).as_posix(),
        'snapshot_sha256':digest,'candidate_audit_id':candidate_id,
        'reviewed_audit_sha256':sha256_file(rpath),
        'mapping_snapshot_path':mapping_copy.relative_to(PROJECT_ROOT).as_posix(),
        'mapping_snapshot_sha256':mapping_sha,
        'source_files':local_source_manifest.to_dict('records'),
        'frozen_historical_sha256':FROZEN_PARQUET_SHA256,
        'historical_overlap_count':len(overlap_report),
        'historical_overlap_max_abs_return_diff':float(overlap_report.abs_return_diff.max()),
        'review_method':'explicit manual attestation; not inferred from bar count'}
    mf=exported_snapshot.with_suffix('.manifest.json')
    print('MANIFEST:',write_once(mf,canonical_bytes(export_manifest)))
    canonical=PROCESSED_ROOT/'es_arena_completed_sessions_v0_1.csv'
    if not canonical.exists():
        print('FIRST CANONICAL NB10 FILE:',write_once(canonical,payload))
    elif sha256_file(canonical)==digest:
        print('CANONICAL NB10 FILE already identical')
    else:
        print('CANONICAL NB10 FILE already exists — NOT OVERWRITTEN.')
        print('For the next NB10 manual preview, explicitly set PREVIEW_SESSIONS_PATH to:')
        print(exported_snapshot)
    print('SNAPSHOT VERIFIED:',sha256_file(exported_snapshot)==digest)
    print('NOTEBOOK 10 INPUT:',exported_snapshot)
    print('NO Arena challenge was fetched; NO forecast submitted.')

DATED SYMBOLOGY PROVENANCE: CREATED
IMMUTABLE SESSION SNAPSHOT: CREATED
MANIFEST: CREATED
CANONICAL NB10 FILE already exists — NOT OVERWRITTEN.
For the next NB10 manual preview, explicitly set PREVIEW_SESSIONS_PATH to:
C:\Users\melny\Market-Mood-Forecasting\data\processed\es_arena_nb11_v0_1\snapshots\es_sessions_20261023_8cf115d4a9a4e666.csv
SNAPSHOT VERIFIED: True
NOTEBOOK 10 INPUT: C:\Users\melny\Market-Mood-Forecasting\data\processed\es_arena_nb11_v0_1\snapshots\es_sessions_20261023_8cf115d4a9a4e666.csv
NO Arena challenge was fetched; NO forecast submitted.


## 13 · Final status and handoff checklist

**Expected first-run outcome:** all safety switches off, a locally calculated request window, local DBN file catalog, synthetic tests passing, and processing/exports skipped. This is a successful SAFE PLAN execution. The saved deployment artifact from Notebook 10 remains untouched.

### Next actions, in order

- [ ] Verify original recent Notebook 08 DBN paths and obtain an incremental quote (only if missing data).
- [ ] Confirm estimated cost with user approval **before** enabling paid download; never rerun a failed paid call without investigation.
- [ ] Assemble 20+ eligible sessions from actual DBN bars (Notebook 08 context + new files).
- [ ] Resolve per-bar date-effective underlying futures symbols and inspect roll transitions.
- [ ] Reconcile at least two actual overlapping recent sessions with the frozen Notebook 08 parquet.
- [ ] Review all session gaps, holidays, coverage and eligibility; manually sign the review CSV.
- [ ] Produce one SHA-256 verified, immutable 18-column CSV for Notebook 10.
- [ ] Obtain an **official** Arena challenge snapshot and verify the actual selected contract and neutral band before a local preview.
- [ ] Open Notebook 10 in a controlled subsequent phase for a **non-submitted** forecast preview; preserve its original frozen deployment artifact.

This notebook **never** calls `headlinearena.com` or a prediction/submission endpoint.

In [14]:
print('='*72)
print('NOTEBOOK 11 — SAFE INCREMENTAL DATA STATUS')
print('Frozen Notebook 08 parquet:',FROZEN_PARQUET.name,'VERIFIED')
print('Incremental quote:', 'REQUESTED' if ENABLE_METADATA_QUOTE else 'DISABLED')
print('Paid market-data download:', 'AUTHORIZED VIA GUARD' if ALLOW_PAID_DOWNLOAD else 'DISABLED')
print('Local DBN input files:',len(selected_files))
print('Resolved minute bars:',0 if resolved_bars is None else len(resolved_bars))
print('Reconstructed session rows:',0 if session_candidates is None else len(session_candidates))
print('Notebook 08 overlap:', 'PASS' if overlap_pass else 'PENDING')
print('Manual coverage attestation:', 'PROVIDED' if REVIEWED_AUDIT_CSV else 'PENDING')
print('Reviewed session export:', exported_snapshot if exported_snapshot else 'DISABLED / PENDING')
print('Arena API forecast submission: NEVER IMPLEMENTED')
print('Notebook 09 consumed holdout: UNTOUCHED')
print('Notebook 10 frozen deployment model: UNTOUCHED')
print('SAFE PLAN / DATA PIPELINE: READY')
print('='*72)

NOTEBOOK 11 — SAFE INCREMENTAL DATA STATUS
Frozen Notebook 08 parquet: es_daily_modeling_v0_1.parquet VERIFIED
Incremental quote: DISABLED
Paid market-data download: DISABLED
Local DBN input files: 6
Resolved minute bars: 82548
Reconstructed session rows: 60
Notebook 08 overlap: PASS
Manual coverage attestation: PROVIDED
Reviewed session export: C:\Users\melny\Market-Mood-Forecasting\data\processed\es_arena_nb11_v0_1\snapshots\es_sessions_20261023_8cf115d4a9a4e666.csv
Arena API forecast submission: NEVER IMPLEMENTED
Notebook 09 consumed holdout: UNTOUCHED
Notebook 10 frozen deployment model: UNTOUCHED
SAFE PLAN / DATA PIPELINE: READY
